# 5.4 Öznitelik Mühendisliği

Bu notebook, PDS Handbook (TR) web sayfasının **Türkçe Jupyter karşılığıdır** — aynı açıklamalar, ders notları ve kod örnekleri.

| | |
|---|---|
| **Web sayfası** | `chapters/05-sklearn/04-feature-engineering.html` |
| **Çalıştırma** | JupyterLab, VS Code veya Colab — hücreleri **yukarıdan aşağı** sırayla (`Shift+Enter`) |
| **Bağımlılık** | Kod hücreleri birbirine bağlıdır; hata alırsanız önce üsttekileri çalıştırın |

> **Kaynak:** Jake VanderPlas, *Python Data Science Handbook* — Türkçe ders uyarlaması



Orijinal: 05.04 Feature Engineering

Önceki bölümler makine öğrenmesinin temel fikirlerini özetledi; ancak tüm örnekler sayısal verinin düzenli [n_samples, n_features] biçiminde olduğunu varsaydı.
    Gerçek dünyada veri nadiren bu biçimde gelir.
    Makine öğrenmesini pratikte kullanmanın daha önemli adımlarından biri öznitelik mühendisliğidir: probleminiz hakkındaki bilgiyi, öznitelik matrisi oluşturmak için kullanabileceğiniz sayılara dönüştürmek.

Bu bölümde öznitelik mühendisliğinin birkaç yaygın örneğini ele alacağız: kategorik veri, metin ve görüntüler için öznitelikler.
    Ayrıca model karmaşıklığını artırmak için türetilmiş öznitelikler ve eksik veri doldurma (imputation) tartışılacak.
    Bu sürece genelde vektörizasyon denir; keyfi veriyi düzgün vektörlere dönüştürmeyi içerir.

## Kategorik Öznitelikler

Sayısal olmayan verinin yaygın türlerinden biri kategorik veridir.
    Örneğin konut fiyatları verisini inceliyorsunuz; "fiyat" ve "oda sayısı" gibi sayısal özniteliklerin yanında "mahalle" bilgisi de vardır.
    Veriniz kabaca şöyle görünebilir:


In [ ]:
# housing_dict_data.py
data = [
    {'price': 850000, 'rooms': 4, 'neighborhood': 'Queen Anne'},
    {'price': 700000, 'rooms': 3, 'neighborhood': 'Fremont'},
    {'price': 650000, 'rooms': 3, 'neighborhood': 'Wallingford'},
    {'price': 600000, 'rooms': 2, 'neighborhood': 'Fremont'}
]



Bu veriyi basit bir sayısal eşlemeyle kodlamak cazip gelebilir:


In [ ]:
# neighborhood_map_bad.py
{'Queen Anne': 1, 'Fremont': 2, 'Wallingford': 3};



Ancak bu Scikit-Learn'de genelde yararlı bir yaklaşım değildir. Paketin modelleri sayısal özniteliklerin cebirsel nicelikleri yansıttığını varsayar; böyle bir eşleme örneğin Queen Anne < Fremont < Wallingford veya Wallingford − Queen Anne = Fremont anlamına gelir ki bu pek anlamlı değildir.

Bu durumda kanıtlanmış tekniklerden biri one-hot encoding (tek-sıcak kodlama)dir: kategorinin varlığını veya yokluğunu 1 veya 0 ile gösteren ek sütunlar oluşturur.
    Veriniz sözlük listesi biçimindeyken Scikit-Learn'ün DictVectorizer sınıfı bunu sizin için yapar:


In [ ]:
# dict_vectorizer.py
from sklearn.feature_extraction import DictVectorizer
vec = DictVectorizer(sparse=False, dtype=int)
vec.fit_transform(data)



neighborhood sütununun üç ayrı mahalle etiketini temsil eden üç sütuna genişletildiğine dikkat edin; her satır kendi mahalle sütununda 1 içerir.
    Kategorik öznitelikler bu şekilde kodlandıktan sonra normal şekilde Scikit-Learn modeli uydurabilirsiniz.

Her sütunun anlamını görmek için öznitelik adlarına bakabilirsiniz:


In [ ]:
# feature_names_out.py
vec.get_feature_names_out()



Bu yaklaşımın belirgin bir dezavantajı vardır: kategorinizin çok olası değeri varsa veri kümenizin boyutu ciddi artabilir.
    Ancak kodlanmış veri çoğunlukla sıfır içerdiğinden seyrek çıktı çok verimli bir çözüm olabilir:


In [ ]:
# dict_vectorizer_sparse.py
vec = DictVectorizer(sparse=True, dtype=int)
vec.fit_transform(data)



Scikit-Learn tahmin edicilerinin neredeyse tamamı uyum ve değerlendirme sırasında böyle seyrek girdileri kabul eder.
    sklearn.preprocessing.OneHotEncoder ve sklearn.feature_extraction.FeatureHasher bu tür kodlamayı destekleyen ek araçlardır.

> **Not**
>

## Metin Öznitelikleri

Öznitelik mühendisliğinde bir başka yaygın ihtiyaç metni temsil edici sayısal değerlere dönüştürmektir.
    Örneğin sosyal medya verisinin otomatik madenciliği büyük ölçüde metnin sayılara kodlanmasına dayanır.
    En basit kodlama yöntemlerinden biri kelime sayımıdır: her metin parçasındaki her kelimenin geçiş sayısını sayıp sonucu tabloya koyarsınız.

Örneğin aşağıdaki üç cümleyi düşünün:


In [ ]:
# text_sample_phrases.py
sample = ['problem of evil',
          'evil queen',
          'horizon problem']



Kelime sayımına dayalı vektörizasyon için "problem", "of", "evil" vb. kelimeleri temsil eden sütunlar oluşturabiliriz.
    Bu basit örnekte elle yapılabilir; tediumdan kaçınmak için Scikit-Learn'ün CountVectorizer sınıfını kullanırız:


In [ ]:
# count_vectorizer.py
from sklearn.feature_extraction.text import CountVectorizer

vec = CountVectorizer()
X = vec.fit_transform(sample)
X



Sonuç her kelimenin kaç kez geçtiğini kaydeden seyrek bir matristir; etiketli sütunlarla DataFrame'e çevirirsek incelemek kolaylaşır:


In [ ]:
# count_vectorizer_df.py
import pandas as pd
pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out())



Basit ham kelime sayımının sorunları vardır: çok sık geçen kelimelere fazla ağırlık verebilir; bazı sınıflandırma algoritmaları için alt optimal olabilir.
    Bunu düzeltmek için term frequency–inverse document frequency (TF–IDF) kullanılır; kelime sayımlarını belgelerde ne sıklıkla göründüklerine göre ağırlıklandırır.
    Bu öznitelikleri hesaplama sözdizimi önceki örneğe benzer:


In [ ]:
# tfidf_vectorizer.py
from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer()
X = vec.fit_transform(sample)
pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out())



TF–IDF'in sınıflandırma probleminde kullanımına Derinlemesine: Naive Bayes Sınıflandırması bölümünde örnek verilmiştir.

### 🧪 Şimdi deneyin

🧪 
      Basit TF–IDF vektörizasyonu deneyin:
          
      from sklearn.feature_extraction.text import TfidfVectorizer
docs = ['problem of evil', 'evil queen', 'problem of heart']
vec = TfidfVectorizer()
X = vec.fit_transform(docs)
print(vec.get_feature_names_out())
print(X.toarray().round(2))

## Görüntü Öznitelikleri

Bir başka yaygın ihtiyaç görüntüleri makine öğrenmesi analizi için uygun biçimde kodlamaktır.
    En basit yaklaşım Scikit-Learn'e Giriş bölümündeki rakam verisinde kullandığımız gibidir: doğrudan piksel değerlerinin kendisi.
    Ancak uygulamaya göre bu optimal olmayabilir.

Görüntüler için öznitelik çıkarma tekniklerinin kapsamlı özeti bu bölümün kapsamını aşar; ancak standart yaklaşımların birçoğu Scikit-Image projesinde mükemmel uygulamalara sahiptir.
    Scikit-Learn ve Scikit-Image birlikte kullanımına Görüntü Öznitelikleri bölümüne bakın.

## Türetilmiş Öznitelikler

Bir başka yararlı öznitelik türü girdi özniteliklerinden matematiksel olarak türetilenlerdir.
    Hiperparametreler ve Model Doğrulama bölümünde girdi verisinden polinom öznitelikleri oluşturduğumuz bir örnek gördük.
    Doğrusal regresyonu polinom regresyona dönüştürmek için modeli değiştirmeden girdiyi dönüştürdük!

Örneğin bu veri düz bir çizgiyle iyi tanımlanamaz (Şekil 40-1):


```python
# derived_features_plot1.py
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

x = np.array([1, 2, 3, 4, 5])
y = np.array([4, 2, 1, 3, 7])
plt.scatter(x, y);
```

*(Jupyter/IPython veya özel ortam gerekir — web sayfasında salt okunur.)*


LinearRegression ile yine de veriye doğru uydurup optimal sonucu alabiliriz (Şekil 40-2):


In [ ]:
# linear_on_nonlinear.py
from sklearn.linear_model import LinearRegression
X = x[:, np.newaxis]
model = LinearRegression().fit(X, y)
yfit = model.predict(X)
plt.scatter(x, y)
plt.plot(x, yfit);



Ancak $x$ ile $y$ arasındaki ilişkiyi tanımlamak için daha gelişmiş bir modele ihtiyacımız olduğu açıktır.

Buna bir yaklaşım veriyi dönüştürüp modele daha fazla esneklik verecek ek öznitelik sütunları eklemektir.
    Örneğin polinom öznitelikleri şöyle ekleyebiliriz:


In [ ]:
# polynomial_features.py
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=3, include_bias=False)
X2 = poly.fit_transform(X)
print(X2)



Türetilmiş öznitelik matrisinde $x$, $x^2$ ve $x^3$'ü temsil eden sütunlar vardır.
    Bu genişletilmiş girdi üzerinde doğrusal regresyon veriye çok daha yakın bir uyum verir (Şekil 40-3):


In [ ]:
# poly_fit_plot.py
model = LinearRegression().fit(X2, y)
yfit = model.predict(X2)
plt.scatter(x, y)
plt.plot(x, yfit);



Modeli değiştirmeden girdileri dönüştürerek modeli iyileştirme fikri birçok güçlü makine öğrenmesi yönteminin temelidir.
    temel fonksiyon regresyonu bağlamında Derinlemesine: Doğrusal Regresyon bölümünde daha derin ineceğiz.
    Daha genel olarak bu, çekirdek yöntemleri olarak bilinen güçlü tekniklere giden motivasyon yollarından biridir; Derinlemesine: Destek Vektör Makineleri bölümünde keşfedeceğiz.

## Eksik Verinin Doldurulması

Öznitelik mühendisliğinde bir başka yaygın ihtiyaç eksik verinin işlenmesidir.
    DataFrame nesnelerinde eksik veriyi Eksik Veri bölümünde ele aldık; eksik değerler sıkça NaN ile işaretlenir.
    Örneğin veri kümemiz şöyle görünebilir:


In [ ]:
# missing_data_array.py
from numpy import nan
X = np.array([[ nan, 0,   3  ],
              [ 3,   7,   9  ],
              [ 3,   5,   2  ],
              [ 4,   nan, 6  ],
              [ 8,   8,   1  ]])
y = np.array([14, 16, -1,  8, -5])



Tipik bir makine öğrenmesi modelini böyle veriye uygularken önce eksik değerleri uygun bir doldurma değeriyle değiştirmemiz gerekir.
    Buna eksik değerlerin imputation (doldurma) denir; stratejiler basit (sütunun ortalamasıyla değiştirme) ile gelişmiş (matris tamamlama veya sağlam model) arasında değişir.

Gelişmiş yaklaşımlar genelde uygulamaya özgüdür; burada derinlemesine girmeyeceğiz.
    Ortalama, medyan veya en sık değerle temel doldurma için Scikit-Learn SimpleImputer sınıfını sağlar:


In [ ]:
# simple_imputer.py
from sklearn.impute import SimpleImputer
imp = SimpleImputer(strategy='mean')
X2 = imp.fit_transform(X)
X2



Sonuç veride iki eksik değerin sütundaki kalan değerlerin ortalamasıyla değiştirildiğini görüyoruz.
    Bu doldurulmuş veri doğrudan örneğin LinearRegression tahmin edicisine verilebilir:


In [ ]:
# impute_then_regress.py
model = LinearRegression().fit(X2, y)
model.predict(X2)



## Öznitelik Pipeline'ları

Önceki örneklerin herhangi birinde dönüşümleri elle yapmak, özellikle birden fazla adımı zincirlemek istediğinizde hızla yorucu olabilir.
    Örneğin şöyle bir işleme pipeline'ı isteyebiliriz:

Bu tür pipeline'ı kolaylaştırmak için Scikit-Learn Pipeline nesnesi sunar:


In [ ]:
# make_pipeline_impute.py
from sklearn.pipeline import make_pipeline

model = make_pipeline(SimpleImputer(strategy='mean'),
                      PolynomialFeatures(degree=2),
                      LinearRegression())



Bu pipeline standart bir Scikit-Learn nesnesi gibi görünür ve davranır; belirtilen tüm adımları girdi verisine uygular:


In [ ]:
# pipeline_fit.py
model.fit(X, y)  # X with missing values, from above
print(y)
print(model.predict(X))



Modelin tüm adımları otomatik uygulanır.
    Basitlik için bu gösterimde modeli eğitildiği veriye uyguladık; bu yüzden sonucu mükemmel tahmin edebildi (ayrıntılı tartışma için Hiperparametreler ve Model Doğrulama bölümüne bakın).

Scikit-Learn pipeline örnekleri için naive Bayes sınıflandırması bölümüne, ayrıca Derinlemesine: Doğrusal Regresyon ve Derinlemesine: Destek Vektör Makineleri bölümlerine bakın.

> **Not**
>
